#  Retrieval-Augmented Generation (RAG) Workflow
 
[Custom Data] → [Chunking] → [Embedding Model] → [Vector Store]

[User Query] → [Retrieve Top‑K] → [LLM + Context] → [Final Answer]

In [10]:
import os
from dotenv import load_dotenv,find_dotenv
_= load_dotenv(find_dotenv())
groq_api_key = os.environ['GROQ_API_KEY']

In [11]:
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings


/var/folders/z8/cxl79vz558q3_ts3_l1n5zj00000gn/T/ipykernel_57646/3692052576.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [12]:
loaded_text = TextLoader('Data/data.txt').load()

text_splitter = CharacterTextSplitter(chunk_size = 1000, chunk_overlap = 0)
chunks = text_splitter.split_documents(loaded_text)

embedding_model = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

vector_db = FAISS.from_documents(chunks,embedding_model)
 

/var/folders/z8/cxl79vz558q3_ts3_l1n5zj00000gn/T/ipykernel_57646/319303387.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')


In [13]:
retriver = vector_db.as_retriever(search_kwargs = {"k":3})
response = retriver.invoke('what is rainwater harwesting')
response

[Document(id='7673ebbe-6f7a-41c5-9dc5-af3c39050b98', metadata={'source': 'Data/data.txt'}, page_content='Water management is essential for every urban community. Cities need water for drinking, sanitation, agriculture, industries, parks, and construction. However, water supplies are often threatened by population growth, pollution, droughts, and inefficient distribution systems. Sustainable water management includes rainwater harvesting, wastewater treatment, leak detection, water recycling, and the protection of lakes, rivers, and groundwater sources.\\n\\n\n\nRainwater harvesting allows buildings to collect and store rainwater for later use. The collected water can be used for gardening, cleaning, toilet flushing, and other non-drinking purposes. Wastewater treatment plants can clean used water so that it can safely return to rivers or be reused in industrial processes. Cities can also reduce water consumption by promoting low-flow taps, efficient toilets, drip irrigation, and drough

## Simple use with Langchain Expression Language LCEL

In [14]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [15]:
template = """Answer a question based only on the following context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
model = ChatGroq(model='llama-3.3-70b-versatile')


In [16]:
def format_docs(docs):
    return '/n/n'.join([d.page_content for d in docs])

chain = (

    {"context":retriver | format_docs, "question": RunnablePassthrough() }
    | prompt
    | model
    | StrOutputParser()

)

In [18]:
response = chain.invoke('what is rainwater harwesting')
print(response)

According to the context, rainwater harvesting is a method that allows buildings to collect and store rainwater for later use. The collected water can be used for non-drinking purposes such as gardening, cleaning, toilet flushing, and other similar activities.
